# OmniOpinion: Traditional Media Conglomerates (TV, Radio, Press) vs. Local Baseline

This notebook analyzes how adding **$M \in \{1, 2, 3, 4\}$ Traditional Media Vehicle Groups** (each operating **Broadcast TV**, **Radio AM/FM**, and **Written Press**) affects:
1. **Public Opinion Shift in the Direction of the Press Center of Mass ($\bar{\mathbf{m}}$):**
   - Directional projection $\text{Shift}_{\text{CM}}(t) = \frac{\bar{\mathbf{x}}(t) \cdot \bar{\mathbf{m}}}{\|\bar{\mathbf{m}}\|_2 \sqrt{d}}$ (*DellaVigna & Kaplan, 2007; Gentzkow & Shapiro, 2006*).
   - Distance to closest vehicle group center of mass $\min_{g} \|\mathbf{x}_i(t) - \mathbf{m}_g\|_2 / \sqrt{d}$.
2. **Polarization & Effective Number of Opinion Clusters:**
   - Esteban-Ray Polarization ($P_{ER}$), Bramson Dispersion Ratio vs. Uniform Random, Sarle's Bimodality ($BC$), and Laakso-Taagepera Effective Clusters ($N_{\text{eff}}^{\text{IPR}}$).
3. **Fact-Driven News Cycle & Brazilian Demographic Access by Modality:**
   - Exogenous facts arriving at rate $\lambda_{\text{fact}}$ (*Mullainathan & Shleifer, 2005*), covered closer to each vehicle's center of mass $\mathbf{m}_g$ with modality-specific lag ($\tau_{\text{radio}}=0, \tau_{\text{tv}}=1, \tau_{\text{press}}=2$, *Leskovec et al., 2009*) and exponential attention decay (*Wu & Huberman, 2007*).
   - Empirical Brazilian demographic reach (*IBGE PNAD Contínua TIC 2022/2023; Kantar IBOPE Inside Video & Inside Audio 2023; Reuters Institute Digital News Report Brazil 2023/2024*).

In [ ]:
from pathlib import Path
import sys
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import seaborn as sns

PROJECT_ROOT = Path.cwd().parent if Path.cwd().name == 'notebooks' else Path.cwd()
if str(PROJECT_ROOT) not in sys.path:
    sys.path.insert(0, str(PROJECT_ROOT))

RESULTS_DIR = PROJECT_ROOT / 'data' / 'results'
FIGURES_DIR = PROJECT_ROOT / 'figures'
FIGURES_DIR.mkdir(parents=True, exist_ok=True)
RESULTS_DIR.mkdir(parents=True, exist_ok=True)

sns.set_theme(style='whitegrid', context='paper', font_scale=1.2)
plt.rcParams['figure.dpi'] = 140
plt.rcParams['savefig.dpi'] = 300

GROUP_LABELS = {
    0: 'Base Model (M=0, No Media)',
    1: '1 Vehicle Group (M=1: TV+Radio+Press)',
    2: '2 Vehicle Groups (M=2: Duopoly)',
    3: '3 Vehicle Groups (M=3: Tripolar)',
    4: '4 Vehicle Groups (M=4: Pluralistic)',
}
GROUP_PALETTE = {
    'Base Model (M=0, No Media)': '#5c6b73',
    '1 Vehicle Group (M=1: TV+Radio+Press)': '#d62728',
    '2 Vehicle Groups (M=2: Duopoly)': '#1f77b4',
    '3 Vehicle Groups (M=3: Tripolar)': '#2ca02c',
    '4 Vehicle Groups (M=4: Pluralistic)': '#9467bd',
}

In [ ]:
from src.engines.simulation_engine import SimulationEngine
from scripts.run_single_simulation import load_merged_config

trad_parquet = RESULTS_DIR / 'all_trad_media_metrics.parquet'
if trad_parquet.exists():
    print(f'Loading consolidated cluster dataset: {trad_parquet}')
    df = pd.read_parquet(trad_parquet)
else:
    print('Generating local multi-seed reference comparison for M = 0, 1, 2, 3, 4 vehicle groups...')
    records = []
    for m_groups in (0, 1, 2, 3, 4):
        for seed_id in range(5):
            cfg = load_merged_config(PROJECT_ROOT / 'configs' / 'traditional_media_cities.yaml')
            cfg['simulation']['n_agents'] = 2500
            cfg['simulation']['n_steps'] = 80
            cfg['media']['n_vehicle_groups'] = m_groups
            cfg['media']['traditional_enabled'] = (m_groups > 0)
            cfg['media']['enabled'] = (m_groups > 0)
            cfg['media']['social_enabled'] = False
            res = SimulationEngine(config=cfg, base_seed=20261030, task_id=seed_id).run()
            for row in res['history']:
                row['n_vehicle_groups'] = m_groups
                row['city_scale'] = 'small_10k'
                records.append(row)
    df = pd.DataFrame(records)

df['group_label'] = df['n_vehicle_groups'].map(GROUP_LABELS)
print(f'Loaded {len(df):,} rows across vehicle groups: {sorted(df["n_vehicle_groups"].unique())}')

In [ ]:
# Figure 1: Shift in Direction of the Press Center of Mass & Closeness to Vehicles
fig, axes = plt.subplots(1, 3, figsize=(18, 5.2), sharex=True)

sns.lineplot(
    data=df, x='step', y='shift_toward_media_cm', hue='group_label',
    palette=GROUP_PALETTE, errorbar='sd', lw=2.3, ax=axes[0]
)
axes[0].axhline(0.0, ls='--', color='gray', alpha=0.6)
axes[0].set_title('A. Opinion Shift Toward Press Center of Mass')
axes[0].set_xlabel('Simulation Step (t)')
axes[0].set_ylabel(r'Projection $(\bar{x}(t) \cdot \bar{m}) / (\|\bar{m}\|_2 \sqrt{d})$')

sns.lineplot(
    data=df, x='step', y='pop_cm_d0', hue='group_label',
    palette=GROUP_PALETTE, errorbar='sd', lw=2.3, ax=axes[1], legend=False
)
axes[1].axhline(0.0, ls='--', color='gray', alpha=0.6)
axes[1].set_title('B. Population Mean Opinion (Dimension 1)')
axes[1].set_xlabel('Simulation Step (t)')
axes[1].set_ylabel(r'Mean Attitude $\bar{x}_{1}(t)$')

sns.lineplot(
    data=df, x='step', y='mean_dist_to_closest_vehicle', hue='group_label',
    palette=GROUP_PALETTE, errorbar='sd', lw=2.3, ax=axes[2], legend=False
)
axes[2].set_title('C. Mean Distance to Closest Media Vehicle Group')
axes[2].set_xlabel('Simulation Step (t)')
axes[2].set_ylabel(r'$\langle \min_g \|x_i(t) - m_g\|_2 / \sqrt{d} \rangle$')

plt.tight_layout()
fig_path1 = FIGURES_DIR / 'fig_trad1_press_center_of_mass_shift.png'
plt.savefig(fig_path1, bbox_inches='tight')
plt.show()
print(f'Saved: {fig_path1}')

In [ ]:
# Figure 2: Effect of M=1,2,3,4 Traditional Media Groups on Polarization & Clusters
fig, axes = plt.subplots(1, 3, figsize=(18, 5.2), sharex=True)

sns.lineplot(
    data=df, x='step', y='esteban_ray', hue='group_label',
    palette=GROUP_PALETTE, errorbar='sd', lw=2.3, ax=axes[0]
)
axes[0].set_title('A. Multidimensional Polarization ($P_{ER}$)')
axes[0].set_xlabel('Simulation Step (t)')
axes[0].set_ylabel('Esteban-Ray Polarization Index')

sns.lineplot(
    data=df, x='step', y='effective_clusters_ipr', hue='group_label',
    palette=GROUP_PALETTE, errorbar='sd', lw=2.3, ax=axes[1], legend=False
)
axes[1].set_title('B. Effective Number of Opinion Clusters ($N_{\mathrm{eff}}^{\mathrm{IPR}}$)')
axes[1].set_xlabel('Simulation Step (t)')
axes[1].set_ylabel(r'$N_{\mathrm{eff}} = 1 / \sum \pi_c^2$')

sns.lineplot(
    data=df, x='step', y='mean_issue_correlation', hue='group_label',
    palette=GROUP_PALETTE, errorbar='sd', lw=2.3, ax=axes[2], legend=False
)
axes[2].set_title('C. Cross-Dimensional Issue Correlation')
axes[2].set_xlabel('Simulation Step (t)')
axes[2].set_ylabel(r'Mean $|\rho_{ab}|$')

plt.tight_layout()
fig_path2 = FIGURES_DIR / 'fig_trad2_polarization_and_clusters_by_vehicles.png'
plt.savefig(fig_path2, bbox_inches='tight')
plt.show()
print(f'Saved: {fig_path2}')

In [ ]:
# Figure 3: Empirical Brazilian Access Curves (TV, Radio, Press) by Age & Socioeconomic Status
ages = np.linspace(16, 90, 100)
age_norm = (ages - 16.0) / 74.0
ses_vals = np.linspace(0.0, 1.0, 100)

# Marginal curves at median SES = 0.5 and median Age = 41.1y
p_tv_age = np.clip(0.24 + 0.48 * (age_norm ** 0.9) - 0.10 * 0.5, 0.05, 0.85)
p_radio_age = np.clip(0.18 + 0.34 * age_norm + 0.14 * np.sin(np.pi * 0.5), 0.05, 0.75)
p_press_age = np.clip(0.02 + 0.42 * (0.5 ** 1.8) + 0.12 * age_norm, 0.01, 0.65)

med_age_norm = (41.1 - 16.0) / 74.0
p_tv_ses = np.clip(0.24 + 0.48 * (med_age_norm ** 0.9) - 0.10 * ses_vals, 0.05, 0.85)
p_radio_ses = np.clip(0.18 + 0.34 * med_age_norm + 0.14 * np.sin(np.pi * ses_vals), 0.05, 0.75)
p_press_ses = np.clip(0.02 + 0.42 * (ses_vals ** 1.8) + 0.12 * med_age_norm, 0.01, 0.65)

# Exponential Decay of Notice Salience by Modality (Wu & Huberman, 2007; Leskovec et al., 2009)
dt_steps = np.arange(0, 15)
decay_radio = np.exp(-0.45 * np.maximum(0, dt_steps - 0)) * (dt_steps >= 0)
decay_tv = np.exp(-0.30 * np.maximum(0, dt_steps - 1)) * (dt_steps >= 1)
decay_press = np.exp(-0.18 * np.maximum(0, dt_steps - 2)) * (dt_steps >= 2)

fig, axes = plt.subplots(1, 3, figsize=(18, 5.0))

axes[0].plot(ages, p_tv_age, lw=2.5, label='Broadcast TV (PNAD/IBOPE)', color='#d62728')
axes[0].plot(ages, p_radio_age, lw=2.5, label='Radio AM/FM (Inside Audio)', color='#1f77b4')
axes[0].plot(ages, p_press_age, lw=2.5, label='Written Press (Reuters/IVC)', color='#2ca02c')
axes[0].set_title('A. Brazilian Media Access by Age (at SES=0.5)')
axes[0].set_xlabel('Agent Age (Years, IBGE 2022)')
axes[0].set_ylabel('Step Access Probability')
axes[0].legend()

axes[1].plot(ses_vals, p_tv_ses, lw=2.5, label='Broadcast TV', color='#d62728')
axes[1].plot(ses_vals, p_radio_ses, lw=2.5, label='Radio AM/FM', color='#1f77b4')
axes[1].plot(ses_vals, p_press_ses, lw=2.5, label='Written Press (Classes A/B skew)', color='#2ca02c')
axes[1].set_title('B. Brazilian Media Access by Socioeconomic Status')
axes[1].set_xlabel('Socioeconomic Status (SES in [0, 1])')
axes[1].set_ylabel('Step Access Probability')
axes[1].legend()

axes[2].step(dt_steps, decay_radio, where='post', lw=2.5, label='Radio (Lag=0, Fast Decay δ=0.45)', color='#1f77b4')
axes[2].step(dt_steps, decay_tv, where='post', lw=2.5, label='TV (Lag=1, Med Decay δ=0.30)', color='#d62728')
axes[2].step(dt_steps, decay_press, where='post', lw=2.5, label='Press (Lag=2, Slow Decay δ=0.18)', color='#2ca02c')
axes[2].set_title('C. Fact-to-Notice Lag & Temporal Decay by Modality')
axes[2].set_xlabel('Steps Since Fact Occurrence (t - t_fact)')
axes[2].set_ylabel('Active Notice Salience w(t)')
axes[2].legend()

plt.tight_layout()
fig_path3 = FIGURES_DIR / 'fig_trad3_brazilian_media_calibration_and_decay.png'
plt.savefig(fig_path3, bbox_inches='tight')
plt.show()
print(f'Saved: {fig_path3}')